# Preprocessing et Feature Engineering

In [31]:
import pandas as pd
import numpy as np

## Date conversion and Data loading

In [32]:
df = pd.read_csv("../data/merged_dataset.csv")
df["delivery_time"] = pd.to_datetime(df['delivery_time'])

## Handling Missing Values

Linear interpolation for weather, as it's a continuous time series

In [33]:
df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)

C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)
C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda x: x.interpolate(method='linear', limit_direction='both')).reset_index(drop=True)
C:\Users\clemm\AppData\Local\Temp\ipykernel_73124\19154904.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby('site_name').apply(lambda

Power is proportional to the cube of wind speed:

In [34]:
df['wind_speed_100m_cube'] = df['wind_speed_100m'] ** 3

## Feature Engineering: Wind Physics

Wind Shear: Difference in speed between 10m and 100m:

In [35]:
df['wind_shear'] = df['wind_speed_100m'] - df['wind_speed_10m']

## Feature Engineering: Circular Encoding (Direction & Time)

Wind Direction (avoiding the 359° to 0° jump)

In [36]:
df['wind_dir_100m_sin'] = np.sin(np.radians(df['wind_direction_100m']))
df['wind_dir_100m_cos'] = np.cos(np.radians(df['wind_direction_100m']))

Time-based features (Daily and Seasonal cycles)

In [37]:
df['hour'] = df['delivery_time'].dt.hour
df['month'] = df['delivery_time'].dt.month

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 11)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 11)

## Target Normalization (Critical for multi-site modeling)
Predicting Load Factor (0 to 1) instead of raw MW

In [38]:
# df['target'] = df['production'] / df['installed_capacity']

Remove columns that would cause data leakage or are redundant

In [39]:
cols_to_drop = [ 'hour', 'month', 'wind_direction_10m', 'wind_direction_100m'] # 'production',
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [40]:
df.head()

,Unnamed: 0,site_name,delivery_time,production,installed_capacity,utilization_rate,wind_speed_10m,wind_speed_100m,wind_gusts_10m,temperature_2m,...,weather_code,sunshine_duration,wind_speed_100m_cube,wind_shear,wind_dir_100m_sin,wind_dir_100m_cos,hour_sin,hour_cos,month_sin,month_cos
0,0,Belwind Phase 1,2023-01-01 00:00:00+00:00,147.7025,171.0,86.375731,14.603082,19.897738,20.7,12.25,...,51.0,0.0,7877.911982,5.294656,-0.633238,-0.773957,0.000000,1.000000,0.0,1.0
1,1,Belwind Phase 1,2023-01-01 01:00:00+00:00,146.1775,171.0,85.483918,16.182089,21.681328,20.8,12.10,...,51.0,0.0,10191.958316,5.499239,-0.608820,-0.793309,0.269797,0.962917,0.0,1.0
2,2,Belwind Phase 1,2023-01-01 02:00:00+00:00,146.1800,171.0,85.485380,17.969420,23.809662,24.1,11.85,...,51.0,0.0,13497.697496,5.840242,-0.751797,-0.659395,0.519584,0.854419,0.0,1.0
3,3,Belwind Phase 1,2023-01-01 03:00:00+00:00,146.5050,171.0,85.675439,14.792228,19.860010,23.9,11.80,...,3.0,0.0,7833.185089,5.067782,-0.760323,-0.649545,0.730836,0.682553,0.0,1.0
4,4,Belwind Phase 1,2023-01-01 04:00:00+00:00,146.6950,171.0,85.786550,15.001333,19.915070,19.7,11.75,...,3.0,0.0,7898.516174,4.913737,-0.753200,-0.657792,0.887885,0.460065,0.0,1.0


In [41]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [42]:
df.to_csv("../data/processed_dataset.csv")